#今日任务-完成线性回归的简洁实现，模型为教科书中例子
第一步生成数据集

In [2]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

In [3]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

调用框架中现有的API来读取数据。 将features和labels作为API的参数传递，并通过数据迭代器指定batch_size。 此外，布尔值is_train表示是否希望数据迭代器对象在每个迭代周期内打乱数据。

In [4]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [5]:
batch_size = 10
data_iter = load_array((features, labels), batch_size)

为了验证是否正常工作，让我们读取并打印第一个小批量样本。这里使用iter构造Python迭代器，并使用next从迭代器中获取第一项。

In [6]:
next(iter(data_iter))

[tensor([[ 0.5017,  0.1581],
         [ 1.0563, -0.1764],
         [-0.8174, -0.1598],
         [ 0.4020,  0.6757],
         [-0.6557,  0.3870],
         [ 0.4257, -1.1180],
         [ 0.4140, -0.9644],
         [ 0.5610,  0.0568],
         [ 1.3531, -0.5910],
         [ 0.1236,  0.5622]]),
 tensor([[4.6706],
         [6.8977],
         [3.0943],
         [2.6976],
         [1.5679],
         [8.8368],
         [8.3302],
         [5.1234],
         [8.9106],
         [2.5268]])]

#定义模型
对应线性公式 y=w₁x₁+w₂x₂+b 导入nn并定义一个2输入1输出的线性预测模型，考虑后续可更改参数适配不同模型

In [7]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

#初始化参数模型
正如我们在构造nn.Linear时指定输入和输出尺寸一样， 现在我们能直接访问参数以设定它们的初始值。 我们通过net[0]选择网络中的第一个图层， 然后使用weight.data和bias.data方法访问参数。 我们还可以使用替换方法normal_和fill_来重写参数值。

In [8]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

#定义损失函数
计算均方误差使用的是MSELoss类，也称为平方L_2范数。 默认情况下，它返回所有样本损失的平均值。

In [9]:
loss = nn.MSELoss()

这里使用小批量随机梯度下降算法，设置lr的值为0.03

In [10]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

#训练
设置训练代码在每个迭代周期里，我们将完整遍历一次数据集（train_data）， 不停地从中获取一个小批量的输入和相应的标签。 对于每一个小批量，我们会进行以下步骤:
  通过调用net(X)生成预测并计算损失l（前向传播）。
  通过进行反向传播来计算梯度。
  通过调用优化器来更新模型参数。
为了更好的衡量训练效果，我们计算每个迭代周期后的损失，并打印它来监控训练过程

In [11]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000151
epoch 2, loss 0.000099
epoch 3, loss 0.000099


下面我们比较生成数据集的真实参数和通过有限数据训练获得的模型参数。 要访问参数，我们首先从net访问所需的层，然后读取该层的权重和偏置并比较估计得到的参数与生成数据集的真实参数。

In [12]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([0.0013, 0.0006])
b的估计误差： tensor([0.0004])
